# TopoConscious — Real BIDS Data Walkthrough

This notebook demonstrates loading real OpenNeuro / BIDS datasets.
Datasets used:
- **Propofol** : [OpenNeuro ds002898](https://openneuro.org/datasets/ds002898) (awake vs anaesthesia)
- **Sleep** : [OpenNeuro ds000201](https://openneuro.org/datasets/ds000201) (REM vs NREM)
- **DoC** : Liège MCS vs UWS dataset (request from lab)

## 1. Download BIDS datasets via DataLad / OpenNeuro

In [ ]:
# Install DataLad if not available
# pip install datalad datalad-osf

# Example: download propofol dataset
import subprocess
# subprocess.run(['datalad', 'install', 'https://github.com/OpenNeuroDatasets/ds002898.git'])
# subprocess.run(['datalad', 'get', 'ds002898/sub-01/func/'])
print('Datasets should be in BIDS format under data/')

## 2. BIDS layout inspection

In [ ]:
from bids import BIDSLayout
import pandas as pd

bids_dir = 'data/ds002898'   # update to your path
try:
    layout = BIDSLayout(bids_dir, validate=False)
    subjects = layout.get_subjects()
    print(f'Found {len(subjects)} subjects: {subjects[:5]}...')
    bold_files = layout.get(suffix='bold', extension='.nii.gz')
    print(f'BOLD files: {len(bold_files)}')
except Exception as e:
    print(f'BIDS layout error (dataset not yet downloaded): {e}')

## 3. Run TopoConscious on a single subject

In [ ]:
from topoconscious import TopoConsciousPipeline

pipe = TopoConsciousPipeline(
    bids_dir='data/ds002898',
    output_dir='results/propofol/',
    window_size=30,
    step=5,
    n_landmarks=200,
    max_homology_dim=2,
    tr=2.0,
    atlas='aal',
)

# Run on one subject only
# result = pipe.run(subject_ids=['sub-01'])
print('Pipeline configured. Uncomment run() line to execute.')

## 4. Load results and validate

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# After running the pipeline:
results_dir = Path('results/propofol/sub-01')
if results_dir.exists():
    p_df = pd.read_csv(results_dir / 'consciousness_probability.csv')
    wt = np.load(results_dir / 'wasserstein_timeline.npy')
    ml_tl = np.load(results_dir / 'ml_timeline.npy')
    
    fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=False)
    axes[0].fill_between(p_df['window'], p_df['p_conscious'], alpha=0.7)
    axes[0].axhline(0.5, color='red', ls='--'); axes[0].set_ylabel('P(conscious)')
    axes[1].plot(wt, color='tomato'); axes[1].set_ylabel('Wasserstein W2')
    axes[2].plot(ml_tl, color='purple'); axes[2].set_ylabel('Müller-Lyer dist')
    axes[2].set_xlabel('Window index')
    plt.suptitle('sub-01 Propofol Results'); plt.tight_layout(); plt.show()
else:
    print('Run the pipeline first to generate results.')

## 5. Batch validation: label awake vs anaesthesia windows

In [ ]:
from topoconscious.validation import ValidationRunner

# Propofol study: first half of scan = awake, second half = anaesthesia
# Labels: 1=awake(conscious), 0=anaesthesia(unconscious)
#
# In a real study you would load the actual condition labels from the
# BIDS events.tsv or a separate labels CSV.

# Example with loaded time series:
# ts_awake = [load_ts(sub, 'awake') for sub in subjects]
# ts_anest = [load_ts(sub, 'anaesthesia') for sub in subjects]
# ts_list = ts_awake + ts_anest
# labels = np.array([1]*len(ts_awake) + [0]*len(ts_anest))
#
# runner = ValidationRunner(output_dir='results/validation/propofol')
# results = runner.evaluate_dataset(ts_list, labels, 'propofol_real')
# runner.plot_roc_curves(results)
print('See validation.py for full ValidationRunner API.')

## 6. Cycle localization on transition windows

In [ ]:
from topoconscious.localization import CycleLocalizer
from topoconscious.topology import PersistenceEngine
import numpy as np

# Example: localize cycles in a single transition window
rng = np.random.default_rng(0)
window = rng.standard_normal((30, 90))   # 30 TRs, 90 AAL regions

engine = PersistenceEngine(max_dim=2, n_landmarks=50)
diagram = engine.compute(window)

localizer = CycleLocalizer(atlas='aal')
result = localizer.localize_with_complex(
    point_cloud=window,
    region_labels=[f'AAL_{i}' for i in range(90)],
    max_edge_length=4.0,
)

print(f"Significant cycles: {len(result['significant_cycles'])}")
print(f"Regions involved: {result['n_regions_involved']}")
for i, (p, regs) in enumerate(zip(result['persistence'], result['regions_per_cycle'])):
    print(f"  Cycle {i}: persistence={p:.3f}, regions={regs}")